In [ ]:
import kagglehub
path = kagglehub.dataset_download("mdrifaturrahman33/levir-cd-change-detection")
print(path)

In [ ]:
from pathlib import Path
ROOT = Path("data/levir_cd/LEVIR-CD+")

TRAIN_A = ROOT/"train/A"
TRAIN_B = ROOT/"train/B"
TRAIN_Y = ROOT/"train/label"

TEST_A  = ROOT/"test/A"
TEST_B  = ROOT/"test/B"
TEST_Y  = ROOT/"test/label"

print(TRAIN_A.exists(), TRAIN_B.exists(), TRAIN_Y.exists())


In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path("data/levir_cd/LEVIR-CD+")
A0 = sorted((ROOT/"train/A").glob("*"))[0]
B0 = sorted((ROOT/"train/B").glob("*"))[0]
Y0 = sorted((ROOT/"train/label").glob("*"))[0]

a = Image.open(A0).convert("RGB")
b = Image.open(B0).convert("RGB")
y = Image.open(Y0).convert("L")

print("A:", A0.name, a.size, "B:", B0.name, b.size, "Y:", Y0.name, y.size)
print("Mask unique values:", np.unique(np.array(y))[:10])

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(a); plt.title("A (t0)"); plt.axis("off")
plt.subplot(1,3,2); plt.imshow(b); plt.title("B (t1)"); plt.axis("off")
plt.subplot(1,3,3); plt.imshow(y, cmap="gray"); plt.title("Label"); plt.axis("off")
plt.show()


In [ ]:
import torch, gc
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()


In [ ]:
!pip -q install torch torchvision tqdm pillow matplotlib scikit-image

import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from tqdm import tqdm

def list_imgs(d):
    exts={".png",".jpg",".jpeg",".tif",".tiff"}
    d = Path(d)
    return sorted([p for p in d.iterdir() if p.suffix.lower() in exts], key=lambda p: p.name)

class LEVIRPairs(Dataset):
    def __init__(self, a_dir, b_dir, y_dir, augment=True):
        self.A = list_imgs(a_dir)
        self.B = list_imgs(b_dir)
        self.Y = list_imgs(y_dir)
        assert len(self.A)==len(self.B)==len(self.Y), (len(self.A),len(self.B),len(self.Y))
        self.augment = augment

    def __len__(self): return len(self.A)

    def __getitem__(self, i):
        from PIL import Image
        a = Image.open(self.A[i]).convert("RGB")
        b = Image.open(self.B[i]).convert("RGB")
        y = Image.open(self.Y[i]).convert("L")

        a = TF.to_tensor(a)
        b = TF.to_tensor(b)
        y = TF.to_tensor(y)
        y = (y > 0.5).float()  # handles 0/255 masks cleanly [web:79]

        if self.augment:
            if random.random() < 0.5:
                a=TF.hflip(a); b=TF.hflip(b); y=TF.hflip(y)
            if random.random() < 0.5:
                a=TF.vflip(a); b=TF.vflip(b); y=TF.vflip(y)
        return a,b,y

# paths
TRAIN_A = ROOT/"train/A"; TRAIN_B = ROOT/"train/B"; TRAIN_Y = ROOT/"train/label"
TEST_A  = ROOT/"test/A";  TEST_B  = ROOT/"test/B";  TEST_Y  = ROOT/"test/label"

# train/val split
A = list_imgs(TRAIN_A); B = list_imgs(TRAIN_B); Y = list_imgs(TRAIN_Y)
idx = list(range(len(A)))
random.seed(42); random.shuffle(idx)
split = int(0.9*len(idx))
tr_i, va_i = idx[:split], idx[split:]

train_ds = LEVIRPairs(TRAIN_A, TRAIN_B, TRAIN_Y, augment=True)
# to use the split without rewriting dataset, we subset via indices:
train_sub = torch.utils.data.Subset(train_ds, tr_i)
val_sub   = torch.utils.data.Subset(LEVIRPairs(TRAIN_A, TRAIN_B, TRAIN_Y, augment=False), va_i)

train_dl = DataLoader(train_sub, batch_size=1, shuffle=True, num_workers=0)
val_dl   = DataLoader(val_sub,   batch_size=1, shuffle=False, num_workers=0)


# model
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class Encoder(nn.Module):
    def __init__(self, in_ch=3, base=32):
        super().__init__()
        self.c1 = ConvBlock(in_ch, base)
        self.c2 = ConvBlock(base, base*2)
        self.c3 = ConvBlock(base*2, base*4)
        self.c4 = ConvBlock(base*4, base*8)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        f1 = self.c1(x); x = self.pool(f1)
        f2 = self.c2(x); x = self.pool(f2)
        f3 = self.c3(x); x = self.pool(f3)
        f4 = self.c4(x)
        return [f1,f2,f3,f4]

class SiamUNet(nn.Module):
    def __init__(self, in_ch=3, base=32):
        super().__init__()
        self.enc = Encoder(in_ch, base)
        self.up3 = nn.ConvTranspose2d(base*8, base*4, 2, stride=2)
        self.d3  = ConvBlock(base*8, base*4)
        self.up2 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2)
        self.d2  = ConvBlock(base*4, base*2)
        self.up1 = nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.d1  = ConvBlock(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x0, x1):
        f0 = self.enc(x0)
        f1 = self.enc(x1)
        d  = [torch.abs(a-b) for a,b in zip(f0,f1)]
        x = d[3]
        x = self.up3(x); x = torch.cat([x, d[2]], dim=1); x = self.d3(x)
        x = self.up2(x); x = torch.cat([x, d[1]], dim=1); x = self.d2(x)
        x = self.up1(x); x = torch.cat([x, d[0]], dim=1); x = self.d1(x)
        return self.out(x)

def dice_loss_with_logits(logits, y, eps=1e-6):
    p = torch.sigmoid(logits)
    num = 2*(p*y).sum(dim=(2,3))
    den = (p+y).sum(dim=(2,3)) + eps
    return (1 - (num/den)).mean()

@torch.no_grad()
def iou_f1(logits, y, thr=0.5, eps=1e-6):
    p = (torch.sigmoid(logits) > thr).float()
    tp = (p*y).sum(dim=(2,3))
    fp = (p*(1-y)).sum(dim=(2,3))
    fn = ((1-p)*y).sum(dim=(2,3))
    iou = (tp/(tp+fp+fn+eps)).mean().item()
    f1  = (2*tp/(2*tp+fp+fn+eps)).mean().item()
    return iou, f1

device = "cpu"
model = SiamUNet(in_ch=3, base=16).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
bce = nn.BCEWithLogitsLoss()

def run_epoch(dl, train=True):
    model.train(train)
    tot_loss=tot_iou=tot_f1=0.0
    n=0
    for a,b,y in tqdm(dl, leave=False):
        a,b,y = a.to(device), b.to(device), y.to(device)
        logits = model(a,b)
        loss = 0.5*bce(logits,y) + 0.5*dice_loss_with_logits(logits,y)
        if train:
            opt.zero_grad()
            loss.backward()
            opt.step()
        iou, f1 = iou_f1(logits,y)
        bs = a.size(0)
        tot_loss += loss.item()*bs; tot_iou += iou*bs; tot_f1 += f1*bs; n += bs
    return tot_loss/n, tot_iou/n, tot_f1/n

for epoch in range(5):
    tr = run_epoch(train_dl, train=True)
    va = run_epoch(val_dl, train=False)
    print(f"epoch {epoch+1}: train loss/iou/f1={tr[0]:.4f}/{tr[1]:.3f}/{tr[2]:.3f} | val loss/iou/f1={va[0]:.4f}/{va[1]:.3f}/{va[2]:.3f}")

torch.save(model.state_dict(), "siam_unet_levir_cdplus.pt")
print("Saved siam_unet_levir_cdplus.pt")


In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
import torch

testA = list_imgs(TEST_A)
testB = list_imgs(TEST_B)
testY = list_imgs(TEST_Y)

@torch.no_grad()
def eval_on_test(thr=0.5):
    model.eval()
    eps = 1e-6
    tot_iou = tot_f1 = 0.0
    n = 0
    for a_path, b_path, y_path in zip(testA, testB, testY):
        a = TF.to_tensor(Image.open(a_path).convert("RGB")).unsqueeze(0).to(device)
        b = TF.to_tensor(Image.open(b_path).convert("RGB")).unsqueeze(0).to(device)
        y = TF.to_tensor(Image.open(y_path).convert("L")).unsqueeze(0).to(device)
        y = (y > 0.5).float()

        logits = model(a,b)
        p = (torch.sigmoid(logits) > thr).float()

        tp = (p*y).sum()
        fp = (p*(1-y)).sum()
        fn = ((1-p)*y).sum()

        iou = (tp/(tp+fp+fn+eps)).item()
        f1  = (2*tp/(2*tp+fp+fn+eps)).item()

        tot_iou += iou; tot_f1 += f1; n += 1

    print(f"Test IoU={tot_iou/n:.3f}, F1={tot_f1/n:.3f}, thr={thr}")

eval_on_test(thr=0.5)


In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

os.makedirs("preds", exist_ok=True)

def predict_mask(a_path, b_path, thr=0.5):
    a = TF.to_tensor(Image.open(a_path).convert("RGB")).unsqueeze(0).to(device)
    b = TF.to_tensor(Image.open(b_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(a,b))[0,0].cpu().numpy()
    mask = (prob > thr).astype(np.uint8) * 255
    return prob, mask

def changes_only(b_path, mask_u8):
    b = np.array(Image.open(b_path).convert("RGB"))
    m = mask_u8 > 0
    out = b.copy()
    out[~m] = 0
    return out

for i in range(5):
    a_path, b_path = testA[i], testB[i]
    prob, m = predict_mask(a_path, b_path, thr=0.5)
    out = changes_only(b_path, m)

    # optional contour overlay
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    overlay = np.array(Image.open(b_path).convert("RGB"))
    cv2.drawContours(overlay, contours, -1, (255,0,0), 2)

    Image.fromarray(m).save(f"preds/{i:03d}_mask.png")
    Image.fromarray(out).save(f"preds/{i:03d}_changes_only.png")
    Image.fromarray(overlay).save(f"preds/{i:03d}_overlay.png")

    # quick inline view
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(overlay); plt.title("Overlay"); plt.axis("off")
    plt.subplot(1,3,2); plt.imshow(out); plt.title("Changes only"); plt.axis("off")
    plt.subplot(1,3,3); plt.imshow(m, cmap="gray"); plt.title("Mask"); plt.axis("off")
    plt.show()


In [ ]:
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
import os

os.makedirs("final_outputs", exist_ok=True)

def predict_full(a_path, b_path, thr=0.5):
    a = TF.to_tensor(Image.open(a_path).convert("RGB")).unsqueeze(0).to(device)
    b = TF.to_tensor(Image.open(b_path).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(a,b))[0,0].cpu().numpy()
    mask = (prob > thr).astype(np.uint8)
    return prob, mask

def contour_overlay(b_path, mask_u8):
    """Day30 with red contour marking of changes"""
    overlay = np.array(Image.open(b_path).convert("RGB"))
    contours, _ = cv2.findContours(mask_u8 * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (255, 0, 0), thickness=3)  # thick red outlines
    return overlay

def stats(mask_bool):
    changed_pixels = mask_bool.sum()
    total_pixels = mask_bool.size
    pct_change = 100 * changed_pixels / total_pixels
    return changed_pixels, pct_change

# generate for 5 examples
test_idx = [0, 5, 10, 15, 20]

for i in test_idx:
    a_path, b_path = testA[i], testB[i]
    
    _, mask = predict_full(a_path, b_path, thr=0.5)
    mask_bool = mask.astype(bool)
    
    a_img = Image.open(a_path).convert("RGB")
    b_img = Image.open(b_path).convert("RGB")
    overlay = contour_overlay(b_path, mask)  # Day30 + red circles/contours
    
    # stats
    changed_px, pct = stats(mask_bool)
    
    # 1-1-1 layout (1 row, 3 columns)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(a_img); axes[0].set_title("Day 1"); axes[0].axis("off")
    axes[1].imshow(b_img); axes[1].set_title("Day 30"); axes[1].axis("off")
    axes[2].imshow(overlay); axes[2].set_title(f"Changes marked\n{changed_px:,} px ({pct:.1f}%)"); axes[2].axis("off")
    
    plt.tight_layout()
    plt.savefig(f"final_outputs/{i:03d}_1-1-1.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Saved final_outputs/*.png")


In [ ]:
import gradio as gr

def ui_predict(day1_img, day30_img, threshold=0.5):
    """Gradio wrapper for your existing functions"""
    if day1_img is None or day30_img is None:
        return None, None, "Upload both images"
    
    _, mask = predict_full(day1_img, day30_img, thr=float(threshold))
    overlay = contour_overlay(day30_img, mask)
    stats = change_stats(mask)
    
    return day1_img, overlay, stats

demo = gr.Interface(
    fn=ui_predict,
    inputs=[
        gr.Image(type="pil", label="Day 1"),
        gr.Image(type="pil", label="Day 30"),
        gr.Slider(0.1, 0.9, value=0.5, label="Threshold")
    ],
    outputs=[
        gr.Image(label="Day 1"),
        gr.Image(label="Changes"),
        gr.Textbox(label="Stats")
    ],
    title="🛰️ Change Detection",
    description="Upload Day1 + Day30 images → see changes in red contours"
)

demo.launch(share=True, inline=True)  # inline=True keeps it in notebook
